# 01 — Standard PDF Invoice Extraction

Approach **1 of 2**: rule-based extraction using `pypdf` and `pdfplumber`.  
No LLM or internet connection required — results are fully deterministic.

| Step | What we do |
|---|---|
| 1 | Load the PDF and inspect metadata |
| 2 | Extract raw text with **pypdf** |
| 3 | Extract higher-fidelity text + tables with **pdfplumber** |
| 4 | Parse structured fields with regex heuristics via `StandardInvoiceExtractor` |
| 5 | Present results in a DataFrame |
| 6 | Export to CSV |

> Place your own invoice PDFs in the `../examples/` folder and update `INVOICE_PATH` below.

## 1 · Import Required Libraries

In [28]:
import importlib
import re
import sys
from pathlib import Path

import pandas as pd
import pdfplumber
import pypdf

# Make the project src importable from the notebook
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import rs_bill_reading.standard_extractor as _std_mod
importlib.reload(_std_mod)
from rs_bill_reading.standard_extractor import InvoiceData, StandardInvoiceExtractor

print("Libraries loaded successfully.")
print(f"pypdf version    : {pypdf.__version__}")
print(f"pdfplumber version: {pdfplumber.__version__}")

Libraries loaded successfully.
pypdf version    : 6.14.2
pdfplumber version: 0.11.10


## 2 · Load and Display PDF Invoice

In [30]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Change this path to point to any invoice PDF inside ../examples/
INVOICE_PATH = PROJECT_ROOT / "examples" / "arabica spa-6181.pdf"
OUTPUT_CSV   = PROJECT_ROOT / "examples" / "sample_invoice_extracted.csv"
# ───────────────────────────────────────────────────────────────────────────────

if not INVOICE_PATH.exists():
    raise FileNotFoundError(
        f"Invoice not found at {INVOICE_PATH}\n"
        "Place a PDF invoice in the examples/ folder and update INVOICE_PATH."
    )

file_size_kb = INVOICE_PATH.stat().st_size / 1024

print(f"File : {INVOICE_PATH.name}")
print(f"Size : {file_size_kb:.1f} KB")

with pypdf.PdfReader(INVOICE_PATH) as reader:
    num_pages = len(reader.pages)
    raw_meta  = dict(reader.metadata or {})

print(f"Pages: {num_pages}")
print("\nPDF Metadata:")
for key, val in raw_meta.items():
    print(f"  {key}: {val}")

File : arabica spa-6181.pdf
Size : 66.8 KB
Pages: 1

PDF Metadata:
  /Creator: Chromium
  /Producer: Skia/PDF m78
  /CreationDate: D:20260706130236+00'00'
  /ModDate: D:20260706130236+00'00'


## 3 · Extract Raw Text

### 3a — pypdf (fast, lightweight)

In [31]:
extractor = StandardInvoiceExtractor(INVOICE_PATH)
pypdf_text = extractor.extract_text_pypdf()

print(f"Characters extracted by pypdf: {len(pypdf_text)}")
print("\n--- First 800 characters ---")
print(pypdf_text[:800])

Characters extracted by pypdf: 889

--- First 800 characters ---
Arabica Spa
Venta al por menor en comercio especializado
Hernando de Aguirre 162 oficina 304
Providencia, Santiago
R.U.T.: 77368986-5
FACTURA ELECTRONICA
Nº 6.181
S.I.I. SANTIAGO
 Fecha Emisión: 01 de julio de 2026      
 SEÑOR(ES) : COMERCIAL CAFETERA SPA  RUT : 76182248-9
 DIRECCION : ALBERTO SOLARI 1400 LOCAL A 242 MALL LA SERENA  COMUNA : LA SERENA 
 GIRO : RESTAURANTES  CIUDAD : LA SERENA 
 VENDEDOR :  FECHA VTO. : 2026-07-31
 CONTACTO :    
CODIGO CANTIDAD DETALLE UNIDAD P. UNITARIO TOTAL
30,00  Cafe verde brasil KG   $ 9.990,00 $ 299.700  
      
      
      
      
      
      
      
      
      
      
      
      
      
      
      
      
      
      
      
      
      
REFERENCIAS Sub-Total $ 0
Descuento % o $ $ 0
Monto Neto $ 299.700
I.V.A. (19%)  $ 56.943
Monto Tota


### 3b — pdfplumber (higher fidelity + table extraction)

In [32]:
plumber_text, tables = extractor.extract_text_pdfplumber()

print(f"Characters extracted by pdfplumber: {len(plumber_text)}")
print(f"Tables found                      : {len(tables)}")
print("\n--- First 800 characters ---")
print(plumber_text[:800])

Characters extracted by pdfplumber: 713
Tables found                      : 1

--- First 800 characters ---
Arabica Spa
R.U.T.: 77368986-5
Venta al por menor en comercio especializado
FACTURA ELECTRONICA
Hernando de Aguirre 162 oficina 304
Nº 6.181
Providencia, Santiago
S.I.I. SANTIAGO
Fecha Emisión: 01 de julio de 2026
SEÑOR(ES) : COMERCIAL CAFETERA SPA RUT : 76182248-9
DIRECCION : ALBERTO SOLARI 1400 LOCAL A 242 MALL LA SERENA COMUNA : LA SERENA
GIRO : RESTAURANTES CIUDAD : LA SERENA
VENDEDOR : FECHA VTO. : 2026-07-31
CONTACTO :
CODIGO CANTIDAD DETALLE UNIDAD P. UNITARIO TOTAL
30,00 Cafe verde brasil KG $ 9.990,00 $ 299.700
REFERENCIAS Sub-Total $ 0
Descuento % o $ $ 0
Monto Neto $ 299.700
I.V.A. (19%) $ 56.943
Monto Total $ 356.643
Timbre Electronico SII
Res. Nº 80 del 2014 - Verifique documento: www.sii.cl


### 3c — Raw tables extracted by pdfplumber

Each table is returned as a list of rows, where each row is a list of cell strings.

In [33]:
for i, table in enumerate(tables, 1):
    print(f"\n=== Table {i} ({len(table)} rows × {len(table[0]) if table else 0} cols) ===")
    df_table = pd.DataFrame(table[1:], columns=table[0] if table else None)
    display(df_table)


=== Table 1 (8 rows × 8 cols) ===


,SEÑOR(ES) : COMERCIAL CAFETERA SPA RUT : 76182248-9\nDIRECCION : ALBERTO SOLARI 1400 LOCAL A 242 MALL LA SERENA COMUNA : LA SERENA\nGIRO : RESTAURANTES CIUDAD : LA SERENA\nVENDEDOR : FECHA VTO. : 2026-07-31\nCONTACTO :,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,CODIGO,CANTIDAD,DETALLE,UNIDAD,P. UNITARIO,NaN,NaN,TOTAL
1,,"30,00",Cafe verde brasil,KG,"$ 9.990,00",NaN,NaN,$ 299.700
2,REFERENCIAS,NaN,NaN,NaN,NaN,Sub-Total,$ 0,NaN
3,NaN,NaN,NaN,NaN,NaN,Descuento % o $,$ 0,NaN
4,NaN,NaN,NaN,NaN,NaN,Monto Neto,$ 299.700,NaN
5,NaN,NaN,NaN,NaN,NaN,I.V.A. (19%),$ 56.943,NaN
6,NaN,NaN,NaN,NaN,NaN,Monto Total,$ 356.643,NaN


## 4 · Parse Invoice Fields with Regex Heuristics

`StandardInvoiceExtractor.extract()` runs both libraries and then applies the regex patterns defined in `_PATTERNS` to populate the structured fields of `InvoiceData`.

In [34]:
invoice_data: InvoiceData = extractor.extract()

HEURISTIC_FIELDS = [
    "invoice_number",
    "invoice_date",
    "due_date",
    "vendor_name",
    "subtotal",
    "tax_amount",
    "total_amount",
]

print("Regex-parsed fields:")
print("-" * 40)
for field in HEURISTIC_FIELDS:
    value = getattr(invoice_data, field)
    status = "✓" if value is not None else "✗ not found"
    print(f"  {field:<20} {value if value is not None else status}")

Regex-parsed fields:
----------------------------------------
  invoice_number       6.181
  invoice_date         01 de julio de 2026
  due_date             2026-07-31
  vendor_name          Arabica Spa
  subtotal             299.700
  tax_amount           56.943
  total_amount         356.643


### 4a — Inspect patterns directly

You can inspect and test each regex pattern against the raw text interactively.

In [36]:
print("Pattern match details:")
print("-" * 60)
for field_name, pattern in StandardInvoiceExtractor._PATTERNS.items():
    match = re.search(pattern, invoice_data.raw_text)
    if match:
        print(f"  {field_name:<20} → '{match.group(1).strip()}'")
        print(f"  {'':20}   matched: '{match.group(0)[:50]}'")
    else:
        print(f"  {field_name:<20} → NO MATCH")

Pattern match details:
------------------------------------------------------------


AttributeError: 'NoneType' object has no attribute 'strip'

## 5 · Structure Extracted Data into a DataFrame

In [37]:
# --- Header / summary fields ---
header_record = {
    "source_file":    INVOICE_PATH.name,
    "invoice_number": invoice_data.invoice_number,
    "invoice_date":   invoice_data.invoice_date,
    "due_date":       invoice_data.due_date,
    "vendor_name":    invoice_data.vendor_name,
    "subtotal":       invoice_data.subtotal,
    "tax_amount":     invoice_data.tax_amount,
    "total_amount":   invoice_data.total_amount,
    "pages":          len(invoice_data.pages),
    "tables_found":   len(invoice_data.tables),
}

df_header = pd.DataFrame([header_record])
df_header

,source_file,invoice_number,invoice_date,due_date,vendor_name,subtotal,tax_amount,total_amount,pages,tables_found
0,arabica spa-6181.pdf,6.181,01 de julio de 2026,2026-07-31,Arabica Spa,299.700,56.943,356.643,1,1


In [38]:
# --- Line-items table (first table found, assumed to be the items grid) ---
if invoice_data.tables:
    items_raw = invoice_data.tables[0]
    if items_raw and len(items_raw) > 1:
        # First row is typically the header
        headers = [str(h).strip() if h else f"col_{i}" for i, h in enumerate(items_raw[0])]
        rows = [
            {headers[j]: (cell.strip() if cell else None) for j, cell in enumerate(row)}
            for row in items_raw[1:]
            if any(cell for cell in row)   # skip fully empty rows
        ]
        df_items = pd.DataFrame(rows)
        print("Line-items table:")
        display(df_items)
    else:
        print("Table found but too short to parse as line items.")
else:
    print("No tables detected — line items may be embedded in plain text.")

Line-items table:


,SEÑOR(ES) : COMERCIAL CAFETERA SPA RUT : 76182248-9\nDIRECCION : ALBERTO SOLARI 1400 LOCAL A 242 MALL LA SERENA COMUNA : LA SERENA\nGIRO : RESTAURANTES CIUDAD : LA SERENA\nVENDEDOR : FECHA VTO. : 2026-07-31\nCONTACTO :,col_1,col_2,col_3,col_4,col_5,col_6,col_7
0,CODIGO,CANTIDAD,DETALLE,UNIDAD,P. UNITARIO,NaN,NaN,TOTAL
1,NaN,"30,00",Cafe verde brasil,KG,"$ 9.990,00",NaN,NaN,$ 299.700
2,REFERENCIAS,NaN,NaN,NaN,NaN,Sub-Total,$ 0,NaN
3,NaN,NaN,NaN,NaN,NaN,Descuento % o $,$ 0,NaN
4,NaN,NaN,NaN,NaN,NaN,Monto Neto,$ 299.700,NaN
5,NaN,NaN,NaN,NaN,NaN,I.V.A. (19%),$ 56.943,NaN
6,NaN,NaN,NaN,NaN,NaN,Monto Total,$ 356.643,NaN


## 6 · Export Results to CSV

In [17]:
df_header.to_csv(OUTPUT_CSV, index=False)
print(f"Exported header fields to: {OUTPUT_CSV}")
print(f"Rows: {len(df_header)}  |  Columns: {len(df_header.columns)}")

# Preview what was written
print("\nPreview:")
display(pd.read_csv(OUTPUT_CSV))

Exported header fields to: /Users/rojobravo/Documents/work/szabo_robert/rs_bill_reading/examples/sample_invoice_extracted.csv
Rows: 1  |  Columns: 10

Preview:


,source_file,invoice_number,invoice_date,due_date,vendor_name,subtotal,tax_amount,total_amount,pages,tables_found
0,sample_invoice.pdf,INV-2024-0042,15/01/2024,15/02/2024,ACME Solutions Ltd.,"3,500.00",805.0,"4,305.00",1,1


---

## 7 · Batch Extraction — All Invoices in `examples/`

Process every PDF found in the `examples/` folder and collect the results into a single DataFrame.

In [18]:
EXAMPLES_DIR = PROJECT_ROOT / "examples"
BATCH_CSV    = EXAMPLES_DIR / "all_invoices_extracted.csv"

pdf_files = sorted(EXAMPLES_DIR.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF(s) to process:\n")
for p in pdf_files:
    print(f"  {p.name}")

Found 6 PDF(s) to process:

  ICB 5657289.pdf
  arabica spa-6181.pdf
  bombo spa-68967.pdf
  colun 30358757.pdf
  icb 5657290.pdf
  sample_invoice.pdf


In [20]:
# Inspect raw text from each real invoice to understand their layout
for pdf_path in pdf_files:
    if pdf_path.name == "sample_invoice.pdf":
        continue
    print(f"\n{'='*70}")
    print(f"FILE: {pdf_path.name}")
    print('='*70)
    ext = StandardInvoiceExtractor(pdf_path)
    text, _ = ext.extract_text_pdfplumber()
    print(text[:1500])


FILE: ICB 5657289.pdf
IMPORTADORA Y ALIMENTOS ICB FOOD SERVICE SPA
R.U.T.: 77965620-9
MAYORISTAS DE VINOS Y BEBIDAS ALCOHÓLICAS Y DE
FANTASÍA FACTURA ELECTRONICA
+56224871900 Nº 5.657.289
MAULEN 240
QUILICURA, QUILICURA S.I.I. SANTIAGO
Fecha Emisión: 01 de julio de 2026
SEÑOR(ES) : COMERCIAL CAFETERA SPA RUT : 76182248-9
DIRECCION : ALBERTO SOLARI N°1400 LOCAL A COMUNA : LA SERENA
GIRO : ACTIVIDADES DE RESTAURANTES Y DE SERVICI CIUDAD : LA SERENA
VENDEDOR : FECHA VTO. : 2026-07-31
CONTACTO :
CODIGO CANTIDAD DETALLE UNIDAD P. UNITARIO TOTAL
104525770 6,00 104525770 HARINA S/POLVO 1 kg UN $ 1.250,00 $ 7.500
104544570 1,00 104544570 PIMENTON TIRAS ROJAS UN $ 8.200,00 $ 6.970
ZANAFOODS 2,9 kg
104544890 4,00 104544890 CHOCLO GRANO IQF 1 KG UN $ 2.700,00 $ 10.800
104547460 3,00 104547460 POROTO VERDE CORTE UN $ 2.350,00 $ 7.050
FRANCES 1 KG
104547470 2,00 104547470 ARVEJAS IQF 1 KG UN $ 2.550,00 $ 5.100
REFERENCIAS Sub-Total $ 0
ORDEN DE COMPRA 0000228693 30-06-2026 Descuento % o $ $ 0
Mont

In [23]:
import re as _re

VENDOR_PAT = StandardInvoiceExtractor._PATTERNS["vendor_name"]

for pdf_path in pdf_files:
    if pdf_path.name == "sample_invoice.pdf":
        continue
    ext  = StandardInvoiceExtractor(pdf_path)
    data = ext.extract()
    m = _re.search(VENDOR_PAT, data.raw_text)
    print(f"\n{'─'*60}\nFILE: {pdf_path.name}")
    print(f"  raw_text[:200]:\n    {repr(data.raw_text[:200])}")
    if m:
        print(f"  match span  : {m.span()}")
        print(f"  groups      : {m.groups()}")


────────────────────────────────────────────────────────────
FILE: ICB 5657289.pdf
  raw_text[:200]:
    'IMPORTADORA Y ALIMENTOS ICB FOOD SERVICE SPA\nMAYORISTAS DE VINOS Y BEBIDAS ALCOHÓLICAS Y DE\nFANTASÍA\n+56224871900\nMAULEN 240\nQUILICURA, QUILICURA\nR.U.T.: 77965620-9\nFACTURA ELECTRONICA\nNº 5.657.289\nS.'
  match span  : (45, 79)
  groups      : (None, 'MAYORISTAS DE VINOS Y BEBIDAS ALCO')

────────────────────────────────────────────────────────────
FILE: arabica spa-6181.pdf
  raw_text[:200]:
    'Arabica Spa\nVenta al por menor en comercio especializado\nHernando de Aguirre 162 oficina 304\nProvidencia, Santiago\nR.U.T.: 77368986-5\nFACTURA ELECTRONICA\nNº 6.181\nS.I.I. SANTIAGO\n Fecha Emisión: 01 de'
  match span  : (12, 36)
  groups      : (None, 'Venta al por menor en co')

────────────────────────────────────────────────────────────
FILE: bombo spa-68967.pdf
  raw_text[:200]:
    'COMERCIALIZADORA BOMBO SPA\nDISTRIBUIDORA DE ALIMENTOS\nTalca sur 101, Bodega 22, Barri

In [25]:
records = []

for pdf_path in pdf_files:
    print(f"Processing: {pdf_path.name} ...", end=" ")
    try:
        ext  = StandardInvoiceExtractor(pdf_path)
        data = ext.extract()
        records.append({
            "source_file":    pdf_path.name,
            "vendor_name":    data.vendor_name,
            "vendor_rut":     data.vendor_rut,
            "customer_name":  data.customer_name,
            "invoice_number": data.invoice_number,
            "invoice_date":   data.invoice_date,
            "due_date":       data.due_date,
            "subtotal":       data.subtotal,
            "tax_amount":     data.tax_amount,
            "total_amount":   data.total_amount,
            "pages":          len(data.pages),
            "tables_found":   len(data.tables),
        })
        print("✓")
    except Exception as exc:
        print(f"✗  {exc}")
        records.append({"source_file": pdf_path.name, "error": str(exc)})

df_batch = pd.DataFrame(records)
print(f"\nDone — {len(df_batch)} invoices processed.")
df_batch

Processing: ICB 5657289.pdf ... ✓
Processing: arabica spa-6181.pdf ... ✓
Processing: bombo spa-68967.pdf ... ✓
Processing: colun 30358757.pdf ... ✓
Processing: icb 5657290.pdf ... ✓
Processing: sample_invoice.pdf ... ✓

Done — 6 invoices processed.


,source_file,vendor_name,vendor_rut,customer_name,invoice_number,invoice_date,due_date,subtotal,tax_amount,total_amount,pages,tables_found
0,ICB 5657289.pdf,IMPORTADORA Y ALIMENTOS ICB FOOD SERVICE SPA,77965620-9,COMERCIAL CAFETERA SPA,5.657.289,01 de julio de 2026,2026-07-31,0,7.110,44.530,1,1
1,arabica spa-6181.pdf,Arabica Spa,77368986-5,COMERCIAL CAFETERA SPA,6.181,01 de julio de 2026,2026-07-31,299.700,56.943,356.643,1,1
2,bombo spa-68967.pdf,COMERCIALIZADORA BOMBO SPA,77557762-2,COMERCIAL CAFETERA SPA,68.967,03 de julio de 2026,2026-07-18,40.336,7.664,48.000,1,1
3,colun 30358757.pdf,COOPERATIVA AGRICOLA Y LECHERA DE LA UNION,81094100-6,SOC. COMERCIAL CAFETERA LTDA.,30.358.757,01 de julio de 2026,2026-07-31,0,26.020,162.967,1,1
4,icb 5657290.pdf,IMPORTADORA Y ALIMENTOS ICB FOOD SERVICE SPA,77965620-9,COMERCIAL CAFETERA SPA,5.657.290,01 de julio de 2026,2026-07-31,0,2.508,15.708,1,1
5,sample_invoice.pdf,ACME Solutions Ltd.,NaN,NaN,INV-2024-0042,15/01/2024,15/02/2024,"3,500.00",805.00,"4,305.00",1,1


In [26]:
df_batch.to_csv(BATCH_CSV, index=False)
print(f"Saved batch results → {BATCH_CSV.name}")
print(f"Rows: {len(df_batch)}  |  Columns: {len(df_batch.columns)}")

Saved batch results → all_invoices_extracted.csv
Rows: 6  |  Columns: 12
